In [4]:
import json
import os
from openai import OpenAI
from jinja2 import Environment, FileSystemLoader

In [ ]:
def render_templates(config_path='jsons/config.json', output_dir='.'):
    """
    Load JSON config and render Jinja2 templates for docker-compose and Dockerfile.
    """
    # Load configuration
    with open(config_path) as cfg_file:
        cfg = json.load(cfg_file)

    # Setup Jinja2 environment
    env = Environment(
        loader=FileSystemLoader('.'),
        trim_blocks=True,
        lstrip_blocks=True
    )

    # Mapping of output filenames to template names
    templates = {
        'docker-compose.yml': 'templates/docker-compose.yml.j2',
        'Dockerfile': 'templates/Dockerfile.j2'
    }

    # Render each template
    for output_name, template_name in templates.items():
        tmpl = env.get_template(template_name)
        rendered = tmpl.render(**cfg)
        output_path = f"{output_dir}/{output_name}"
        with open(output_path, 'w') as out_file:
            out_file.write(rendered)
        print(f"Generated {output_path}")


if __name__ == '__main__':
    render_templates()

Generated ./docker-compose.yml
Generated ./Dockerfile


In [ ]:
def generate_tools_class(
    config_path: str = 'config.json',
    template_path: str = 'templates/tools_class.py.j2',
    output_path: str = 'scripts/tools_class.py'
):
    """
    Load JSON config and render the tools_class.py template into a real Python file.
    """
    # 1) Load configuration
    with open(config_path, 'r') as cfg_file:
        cfg = json.load(cfg_file)

    # 2) Prepare Jinja2
    env = Environment(
        loader=FileSystemLoader('.'),
        trim_blocks=True,
        lstrip_blocks=True,
    )
    template = env.get_template(template_path)

    # 3) Render and write
    rendered = template.render(**cfg)
    with open(output_path, 'w') as out_file:
        out_file.write(rendered)

    print(f"Generated ./{output_path}")

if __name__ == '__main__':
    generate_tools_class()

Generated ./scripts/tools_class.py


In [ ]:
client = OpenAI()

def create_vector_store(config_path: str = 'jsons/config.json'):
    # Check if the config.json file has vector store enabled
    if not os.path.exists(config_path):
        print(f"Configuration file '{config_path}' not found.")
        return None

    with open(config_path, 'r') as cfg_file:
        cfg = json.load(cfg_file)

    if not cfg.get('vector_store', {}).get('enabled', False):
        print("Vector store is not enabled in the configuration.")
        return None

    # Create the vector store
    vector_store = client.vector_stores.create(name=cfg['project_name'])

    # Prepare files for upload
    files_dir = os.path.join(os.getcwd(), 'files')
    file_paths = [
        os.path.join(files_dir, fname)
        for fname in os.listdir(files_dir)
        if os.path.isfile(os.path.join(files_dir, fname))
    ]
    file_streams = [open(path, "rb") for path in file_paths]

    # Upload and poll
    file_batch = client.vector_stores.file_batches.upload_and_poll(
        vector_store_id=vector_store.id,
        files=file_streams
    )

    print(f"Uploaded {file_batch.file_counts} files to vector store {vector_store.id}.")
    return vector_store.id

if __name__ == '__main__':
    create_vector_store()

Uploaded VectorStoreFileBatch(id='vsfb_c1cd905fc5084e698ffef3d86c6947b4', created_at=1749491818, file_counts=FileCounts(cancelled=0, completed=1, failed=0, in_progress=0, total=1), object='vector_store.file_batch', status='completed', vector_store_id='vs_68472067da80819195a652521920d1e3') files to vector store vs_68472067da80819195a652521920d1e3.


In [5]:
def generate_functions_class(
    config_path: str = 'jsons/config.json',
    template_path: str = 'templates/functions_class.py.j2',
    output_path: str = 'scripts/functions_class.py'
):
    """
    Load JSON config and render the functions_class.py template into a real Python file.
    """
    # 1) Load configuration
    with open(config_path, 'r') as cfg_file:
        cfg = json.load(cfg_file)

    # 2) Prepare Jinja2 environment
    env = Environment(
        loader=FileSystemLoader(os.getcwd()),
        trim_blocks=True,
        lstrip_blocks=True,
    )
    template = env.get_template(template_path)

    # 3) Render template with your config
    rendered = template.render(**cfg)

    # 4) Ensure output directory exists
    out_dir = os.path.dirname(output_path)
    os.makedirs(out_dir, exist_ok=True)

    # 5) Write the rendered Python class file
    with open(output_path, 'w') as out_file:
        out_file.write(rendered)

    print(f"Generated ./{output_path}")

if __name__ == '__main__':
    generate_functions_class()

Generated ./scripts/functions_class.py


In [ ]:
def generate_assistant(
    config_path: str   = 'jsons/config.json',
    template_path: str = 'templates/assistant_class.py.j2',
    output_path: str   = 'scripts/assistant_class.py'
):
    """
    Load JSON config and render the assistant.py template into a real Python file.
    """
    # 1) Load configuration
    with open(config_path, 'r') as cfg_file:
        cfg = json.load(cfg_file)

    # 2) Prepare Jinja2 environment
    env = Environment(
        loader=FileSystemLoader(os.getcwd()),
        trim_blocks=True,
        lstrip_blocks=True,
    )

    # 3) Get the assistant template
    template = env.get_template(template_path)

    # 4) Render template with your config
    rendered = template.render(**cfg)

    # 5) Ensure output directory exists
    out_dir = os.path.dirname(output_path)
    os.makedirs(out_dir, exist_ok=True)

    # 6) Write the rendered Assistant class file
    with open(output_path, 'w') as out_file:
        out_file.write(rendered)

    print(f"Generated ./{output_path}")

if __name__ == '__main__':
    generate_assistant()

Generated ./scripts/assistant_class.py


In [5]:
def generate_init(
    output_path: str = 'scripts/__init__.py'
):
    """
    Generate an empty __init__.py file in the scripts directory.
    """
    # Ensure the scripts directory exists
    os.makedirs(os.path.dirname(output_path), exist_ok=True)

    # Write an empty __init__.py file
    with open(output_path, 'w') as out_file:
        out_file.write("# This is an init file for the scripts package.\n")

    print(f"Generated ./{output_path}")

if __name__ == '__main__':
    generate_init()

Generated ./scripts/__init__.py


In [ ]:
def generate_main(
    config_path: str   = 'jsons/config.json',
    template_path: str = 'templates/main.py.j2',
    output_path: str   = 'main.py'
):
    """
    Load JSON config and render the main.py template into a real Python entrypoint.
    """
    # 1) Load configuration
    with open(config_path, 'r') as cfg_file:
        cfg = json.load(cfg_file)

    # 2) Prepare Jinja2 environment
    env = Environment(
        loader=FileSystemLoader(os.getcwd()),
        trim_blocks=True,
        lstrip_blocks=True,
    )

    # 3) Get the main.py template
    template = env.get_template(template_path)

    # 4) Render template with your config
    rendered = template.render(**cfg)

    # 5) Ensure output directory exists
    out_dir = os.path.dirname(output_path) or '.'
    os.makedirs(out_dir, exist_ok=True)

    # 6) Write the rendered main.py
    with open(output_path, 'w') as out_file:
        out_file.write(rendered)

    print(f"Generated ./{output_path}")

if __name__ == '__main__':
    generate_main()

Generated ./main.py


In [ ]:
def create_frontend(
    config_path: str   = 'jsons/config.json',
    template_path: str = 'templates/app.py.j2',
    output_path: str   = 'app.py'
):
    """
    Load JSON config and render the app.py template into a real Streamlit frontend.
    """
    # 1) Load configuration
    with open(config_path, 'r') as cfg_file:
        cfg = json.load(cfg_file)

    # 2) Prepare Jinja2 environment
    env = Environment(
        loader=FileSystemLoader(os.getcwd()),
        trim_blocks=True,
        lstrip_blocks=True,
    )

    # 3) Get the app.py template
    template = env.get_template(template_path)

    # 4) Render template with your config
    rendered = template.render(**cfg)

    # 5) Ensure output directory exists
    out_dir = os.path.dirname(output_path) or '.'
    os.makedirs(out_dir, exist_ok=True)

    # 6) Write the rendered app.py
    with open(output_path, 'w') as out_file:
        out_file.write(rendered)

    print(f"Generated ./{output_path}")

if __name__ == '__main__':
    create_frontend()